# EasyVisa – Visa Approval Prediction Project

## 1. Problem Definition
### Context
The Immigration and Nationality Act (INA) of the US permits foreign workers to come to the United States to work on either a temporary or permanent basis. The immigration programs are administered by the Office of Foreign Labor Certification (OFLC). As the number of applicants increases every year, the process of reviewing every case is becoming a tedious task.

### Objective
The objective of this project is to build a Machine Learning classification model that can:
1. Facilitate the process of visa approvals.
2. Recommend suitable profiles for applicants whose visas should be certified or denied based on the drivers that significantly influence the case status.

### Data Dictionary
- `case_id`: ID of each visa application
- `continent`: Continent of the employee
- `education_of_employee`: Education of the employee
- `has_job_experience`: Y = Yes; N = No
- `requires_job_training`: Y = Yes; N = No
- `no_of_employees`: Number of employees in the employer's company
- `yr_of_estab`: Year in which the employer's company was established
- `region_of_employment`: Intended region of employment in the US
- `prevailing_wage`: Average wage paid to similarly employed workers
- `unit_of_wage`: Unit of prevailing wage (Hourly, Weekly, Monthly, Yearly)
- `full_time_position`: Y = Full-Time Position; N = Part-Time Position
- `case_status`: Flag indicating if the Visa was certified or denied (Target Variable)

## 2. Import Libraries and Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, 
    recall_score, 
    precision_score, 
    f1_score, 
    confusion_matrix, 
    classification_report
)

# Classification Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    BaggingClassifier, 
    AdaBoostClassifier, 
    GradientBoostingClassifier
)
from xgboost import XGBClassifier

# Sampling techniques
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

sns.set_theme(style='whitegrid')

In [ ]:
# Load the dataset
data = pd.read_csv('EasyVisa.csv')
df = data.copy()
print(f'Dataset Shape: {df.shape}')
df.head()

## 3. Helper Functions

In [ ]:
def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2, sharex=True, gridspec_kw={'height_ratios': (0.25, 0.75)}, figsize=figsize,
    )
    sns.boxplot(data=data, x=feature, ax=ax_box2, showmeans=True, color='violet')
    sns.histplot(data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins, palette='winter') if bins else sns.histplot(data=data, x=feature, kde=kde, ax=ax_hist2)
    ax_hist2.axvline(data[feature].mean(), color='green', linestyle='--')
    ax_hist2.axvline(data[feature].median(), color='black', linestyle='-')

def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top
    """
    total = len(data[feature])
    count = data[feature].nunique()
    plt.figure(figsize=(count + 1 if n is None else n + 1, 5))
    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(data=data, x=feature, palette='Paired', order=data[feature].value_counts().index[:n])
    for p in ax.patches:
        label = '{:.1f}%'.format(100 * p.get_height() / total) if perc else p.get_height()
        ax.annotate(label, (p.get_x() + p.get_width() / 2, p.get_height()), ha='center', va='center', size=12, xytext=(0, 5), textcoords='offset points')
    plt.show()

def stacked_barplot(data, predictor, target):
    """
    Stacked bar chart for bivariate analysis
    """
    count = data[predictor].nunique()
    sorter = data[target].value_counts().index[-1]
    tab1 = pd.crosstab(data[predictor], data[target], margins=True).sort_values(by=sorter, ascending=False)
    print(tab1)
    print('-' * 120)
    tab = pd.crosstab(data[predictor], data[target], normalize='index').sort_values(by=sorter, ascending=False)
    tab.plot(kind='bar', stacked=True, figsize=(count + 5, 5))
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.show()

def model_performance_classification_sklearn(model, predictors, target):
    """
    Compute classification metrics
    """
    pred = model.predict(predictors)
    acc = accuracy_score(target, pred)
    recall = recall_score(target, pred)
    precision = precision_score(target, pred)
    f1 = f1_score(target, pred)
    return pd.DataFrame({'Accuracy': acc, 'Recall': recall, 'Precision': precision, 'F1': f1}, index=[0])

def confusion_matrix_sklearn(model, predictors, target):
    """
    Plot confusion matrix
    """
    y_pred = model.predict(predictors)
    cm = confusion_matrix(target, y_pred)
    labels = np.asarray([['{0:0.0f}'.format(item) + '\n{0:.2%}'.format(item / cm.flatten().sum())] for item in cm.flatten()]).reshape(2, 2)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt='')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

## 4. Exploratory Data Analysis (EDA)

### 4.1 Data Overview

In [ ]:
df.info()
print('\nMissing Values:')
print(df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())

### 4.2 Univariate Analysis

In [ ]:
labeled_barplot(df, 'case_status', perc=True)
labeled_barplot(df, 'continent', perc=True)
labeled_barplot(df, 'education_of_employee', perc=True)

In [ ]:
histogram_boxplot(df, 'no_of_employees')
histogram_boxplot(df, 'prevailing_wage')

### 4.3 Bivariate Analysis

In [ ]:
stacked_barplot(df, 'education_of_employee', 'case_status')
stacked_barplot(df, 'has_job_experience', 'case_status')

## 5. Data Pre-processing

In [ ]:
# Encoding Target Variable
df['case_status'] = df['case_status'].map({'Certified': 1, 'Denied': 0})

# Dropping unnecessary columns
df_proc = df.drop(['case_id'], axis=1)

# Mapping Categorical Variables
df_proc['continent'] = df_proc['continent'].replace({'Asia': 1, 'Europe': 2, 'North America': 3, 'South America': 4, 'Africa': 5, 'Oceania': 6})
df_proc['education_of_employee'] = df_proc['education_of_employee'].replace({'High School': 1, "Bachelor's": 2, "Master's": 3, 'Doctorate': 4})
df_proc['has_job_experience'] = df_proc['has_job_experience'].replace({'N': 0, 'Y': 1})
df_proc['requires_job_training'] = df_proc['requires_job_training'].replace({'N': 0, 'Y': 1})
df_proc['region_of_employment'] = df_proc['region_of_employment'].replace({'Northeast': 1, 'West': 2, 'South': 3, 'Midwest': 4, 'Island': 5})
df_proc['unit_of_wage'] = df_proc['unit_of_wage'].replace({'Hour': 1, 'Week': 2, 'Month': 3, 'Year': 4})
df_proc['full_time_position'] = df_proc['full_time_position'].replace({'N': 0, 'Y': 1})

# Splitting Data
X = df_proc.drop(['case_status'], axis=1)
y = df_proc['case_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')


## 6. Model Building - Original Data

In [ ]:
def build_and_eval(X_t, y_t, X_v, y_v):
    models = {
        'Decision Tree': DecisionTreeClassifier(random_state=1),
        'Random Forest': RandomForestClassifier(random_state=1),
        'Bagging': BaggingClassifier(random_state=1),
        'AdaBoost': AdaBoostClassifier(random_state=1),
        'Gradient Boosting': GradientBoostingClassifier(random_state=1)
    }
    results = []
    for name, model in models.items():
        model.fit(X_t, y_t)
        perf = model_performance_classification_sklearn(model, X_v, y_v)
        perf.index = [name]
        results.append(perf)
    return pd.concat(results)

res_original = build_and_eval(X_train, y_train, X_test, y_test)
res_original


## 7. Model Building - Oversampled Data (SMOTE)

In [ ]:
sm = SMOTE(random_state=1)
X_train_over, y_train_over = sm.fit_resample(X_train, y_train)

res_over = build_and_eval(X_train_over, y_train_over, X_test, y_test)
res_over


## 8. Model Building - Undersampled Data

In [ ]:
rus = RandomUnderSampler(random_state=1)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

res_under = build_and_eval(X_train_under, y_train_under, X_test, y_test)
res_under


## 9. Hyperparameter Tuning

In [ ]:
# Tuning Random Forest
rf = RandomForestClassifier(random_state=1)
param_grid_rf = {
    'n_estimators': [100, 150],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 5]
}
grid_rf = GridSearchCV(rf, param_grid_rf, cv=3, scoring='f1', n_jobs=-1)
grid_rf.fit(X_train_over, y_train_over)
best_rf = grid_rf.best_estimator_

# Tuning Gradient Boosting
gb = GradientBoostingClassifier(random_state=1)
param_grid_gb = {
    'n_estimators': [100, 150],
    'learning_rate': [0.1, 0.2],
    'max_depth': [3, 5]
}
grid_gb = GridSearchCV(gb, param_grid_gb, cv=3, scoring='f1', n_jobs=-1)
grid_gb.fit(X_train_over, y_train_over)
best_gb = grid_gb.best_estimator_

print(f'Best RF Params: {grid_rf.best_params_}')
print(f'Best GB Params: {grid_gb.best_params_}')


## 10. Model Performances and Selection

In [ ]:
# Comparing Tuned Models on Test Data
perf_rf = model_performance_classification_sklearn(best_rf, X_test, y_test)
perf_gb = model_performance_classification_sklearn(best_gb, X_test, y_test)

final_comparison = pd.concat([perf_rf, perf_gb])
final_comparison.index = ['Tuned Random Forest', 'Tuned Gradient Boosting']
print(final_comparison)

print('\nFinal Model Confusion Matrix (Tuned GB):')
confusion_matrix_sklearn(best_gb, X_test, y_test)


## 11. Actionable Business Insights and Recommendations

- **Education and Experience:** Applicants with higher education (Master's/Doctorate) and job experience are significantly more likely to have their visas certified.
- **Wages:** Higher prevailing wages correlate with higher certification rates, suggesting that premium skill positions are prioritized.
- **Recommendations:** OFLC should streamline the application process for candidates meeting these criteria while conducting more thorough reviews for those with less experience or lower education levels.